# The CUDA Programming Model

Companion notebook for the [CUDA Programming Model lesson](https://ml-viz.vercel.app/courses/gpu-programming/02-cuda-programming-model).

We **emulate** the CUDA execution model in pure Python: a kernel is a per-thread function, and we
drive it across a grid of blocks and threads exactly as the GPU would — computing each thread's
global index, applying the bounds guard, and finally using the **grid-stride loop**. No GPU needed;
the point is the indexing mental model that underlies PyTorch, Triton, and friends.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np

## 1 — A kernel is a per-thread function

In CUDA you write the body for *one* thread; the runtime launches thousands. We mimic that: a
`launch` driver loops over every (block, thread) pair and calls the kernel with that thread's
coordinates. The kernel computes its **global index** and writes one output element.

In [ ]:
def launch(kernel, grid_dim, block_dim, *args):
    """Emulate <<<grid_dim, block_dim>>> by calling `kernel` once per thread."""
    for block_idx in range(grid_dim):
        for thread_idx in range(block_dim):
            kernel(block_idx, block_dim, thread_idx, *args)

def vec_add(blockIdx, blockDim, threadIdx, A, B, C, n):
    i = blockIdx * blockDim + threadIdx        # the global-index formula
    if i < n:                                  # bounds guard for over-launch
        C[i] = A[i] + B[i]

n = 1000
A = np.arange(n, dtype=np.float32)
B = np.arange(n, dtype=np.float32) * 10
C = np.zeros(n, dtype=np.float32)

threads = 256
blocks = (n + threads - 1) // threads          # ceil(n / threads)
print(f"launching {blocks} blocks x {threads} threads = {blocks*threads} threads for n={n}")
launch(vec_add, blocks, threads, A, B, C, n)

assert np.allclose(C, A + B)
print("\u2713 vec_add kernel matches A + B")

## 2 — Why the bounds guard matters

We launched `blocks*threads = 1024` threads for only `n = 1000` elements. The last 24 threads have
`i >= n`. Without `if (i < n)` they would write out of bounds. Let's see exactly which threads idle.

In [ ]:
active, idle = 0, 0
def count_kernel(blockIdx, blockDim, threadIdx, n):
    global active, idle
    i = blockIdx * blockDim + threadIdx
    if i < n:
        active += 1
    else:
        idle += 1

launch(count_kernel, blocks, threads, n)
print(f"launched threads: {blocks*threads}")
print(f"active (i < n):   {active}")
print(f"idle   (i >= n):  {idle}   <- these would corrupt memory without the guard")

## 3 — The grid-stride loop

Hard-coding one element per thread couples the launch size to `n`. The **grid-stride loop** lets each
thread handle multiple elements, striding by the total number of threads in the grid — so *any* grid
size is correct. We launch a deliberately small grid and still cover all `n` elements.

In [ ]:
def vec_add_stride(blockIdx, blockDim, threadIdx, gridDim, A, B, C, n):
    stride = blockDim * gridDim                # total threads in the grid
    i = blockIdx * blockDim + threadIdx
    while i < n:
        C[i] = A[i] + B[i]
        i += stride

C2 = np.zeros(n, dtype=np.float32)
small_blocks, small_threads = 4, 32            # only 128 threads for n=1000!
for b in range(small_blocks):
    for t in range(small_threads):
        vec_add_stride(b, small_threads, t, small_blocks, A, B, C2, n)

assert np.allclose(C2, A + B)
print(f"{small_blocks*small_threads} threads covered all {n} elements via grid-stride")
print(f"each thread handled ~{n // (small_blocks*small_threads)} elements")

## ✏️ Your turn

**Exercise.** Implement `global_indices(grid_dim, block_dim, n)` returning the list of global indices
`i = blockIdx*blockDim + threadIdx` for **only the active threads** (those with `i < n`), in launch
order. This is the set of elements a one-element-per-thread kernel actually processes.

In [ ]:
def global_indices(grid_dim, block_dim, n):
    out = []
    for blockIdx in range(grid_dim):
        for threadIdx in range(block_dim):
            # TODO(you): compute the global index and append it only if it is in bounds
            ...
    return out

In [ ]:
# This assert cell passes silently when your implementation is correct.
idx = global_indices(4, 256, 1000)
assert idx == list(range(1000)), "active threads should cover 0..999 exactly once"
assert global_indices(1, 8, 5) == [0, 1, 2, 3, 4]          # last 3 threads idle
assert len(global_indices(4, 256, 1000)) == 1000            # 24 of the 1024 threads idle
print("\u2713 global index mapping is correct, bounds guard drops the over-launched threads")

<details>
<summary>Solution</summary>

```python
def global_indices(grid_dim, block_dim, n):
    out = []
    for blockIdx in range(grid_dim):
        for threadIdx in range(block_dim):
            i = blockIdx * block_dim + threadIdx
            if i < n:
                out.append(i)
    return out
```

Every framework that dispatches to the GPU computes this same mapping under the hood. When you write
`a + b` in PyTorch, a pre-written CUDA kernel runs exactly this index math across the tensor.

</details>